<a href="https://colab.research.google.com/github/Sakeena01/AI-Assisted-Threat-Detection-Dashboard/blob/main/SQL_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# 1. Load the SQL tool into Colab
%load_ext sql

# 2. Fix the Google Colab display bug (prevents KeyError)
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

# 3. Connect to a temporary database
%sql sqlite://

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [10]:
%%sql
-- Drop the table if it already exists to avoid errors
DROP TABLE IF EXISTS online_orders;

-- Now create the clean target table safely
CREATE TABLE online_orders (
    order_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    category TEXT,
    order_amount REAL,
    order_date DATE,
    shipping_country TEXT
);

-- Populate the table with mock transactions
INSERT INTO online_orders (customer_name, category, order_amount, order_date, shipping_country) VALUES
('John Doe', 'Electronics', 1200.50, '2026-01-15', 'USA'),
('Jane Smith', 'Furniture', 450.00, '2026-01-18', 'Canada'),
('Rahul Kumar', 'Electronics', 85.00, '2026-02-02', 'India'),
('Anna Miller', 'Apparel', 45.25, '2026-02-10', 'USA'),
('John Doe', 'Apparel', 120.00, '2026-02-14', 'USA'),
('Carlos Ruiz', 'Furniture', 899.00, '2026-03-01', 'Mexico'),
('Jane Smith', 'Electronics', 60.00, '2026-03-05', 'Canada'),
('Rahul Kumar', 'Apparel', 35.00, '2026-03-12', 'India');


 * sqlite://
Done.
Done.
8 rows affected.


[]

1. Get premium orders sorted by cost.

In [11]:
%%sql
SELECT order_id, customer_name, order_amount, category
FROM online_orders
WHERE order_amount >= 100.00
ORDER BY order_amount DESC;


 * sqlite://
Done.


order_id,customer_name,order_amount,category
1,John Doe,1200.5,Electronics
6,Carlos Ruiz,899.0,Furniture
2,Jane Smith,450.0,Furniture
5,John Doe,120.0,Apparel


2. Find international orders from specific regions.

In [12]:
%%sql
SELECT *
FROM online_orders
WHERE shipping_country IN ('Canada', 'India', 'Mexico')
  AND customer_name LIKE 'J%'; -- Finds names starting with 'J'


 * sqlite://
Done.


order_id,customer_name,category,order_amount,order_date,shipping_country
2,Jane Smith,Furniture,450.0,2026-01-18,Canada
7,Jane Smith,Electronics,60.0,2026-03-05,Canada


3. Segment orders based on their financial tier.

In [13]:
%%sql
SELECT
    order_id,
    customer_name,
    order_amount,
    CASE
        WHEN order_amount >= 500.00 THEN 'High-Value Sale'
        WHEN order_amount >= 100.00 THEN 'Mid-Value Sale'
        ELSE 'Low-Value Sale'
    END AS order_segment
FROM online_orders;


 * sqlite://
Done.


order_id,customer_name,order_amount,order_segment
1,John Doe,1200.5,High-Value Sale
2,Jane Smith,450.0,Mid-Value Sale
3,Rahul Kumar,85.0,Low-Value Sale
4,Anna Miller,45.25,Low-Value Sale
5,John Doe,120.0,Mid-Value Sale
6,Carlos Ruiz,899.0,High-Value Sale
7,Jane Smith,60.0,Low-Value Sale
8,Rahul Kumar,35.0,Low-Value Sale


4. Target major spending accounts with multiple transactions.


In [14]:
%%sql
SELECT
    customer_name,
    COUNT(order_id) AS total_orders,
    SUM(order_amount) AS total_spent,
    ROUND(AVG(order_amount), 2) AS average_order_value
FROM online_orders
GROUP BY customer_name
HAVING COUNT(order_id) >= 2; -- Excludes customers with only 1 order


 * sqlite://
Done.


customer_name,total_orders,total_spent,average_order_value
Jane Smith,2,510.0,255.0
John Doe,2,1320.5,660.25
Rahul Kumar,2,120.0,60.0


5. Geographical & Market Segmentation

In [15]:
%%sql
SELECT
    order_id,
    customer_name,
    shipping_country,
    CASE
        WHEN shipping_country IN ('USA', 'Canada', 'Mexico') THEN 'North America'
        WHEN shipping_country = 'India' THEN 'Asia'
        ELSE 'Other Region'
    END AS continent_group
FROM online_orders;


 * sqlite://
Done.


order_id,customer_name,shipping_country,continent_group
1,John Doe,USA,North America
2,Jane Smith,Canada,North America
3,Rahul Kumar,India,Asia
4,Anna Miller,USA,North America
5,John Doe,USA,North America
6,Carlos Ruiz,Mexico,North America
7,Jane Smith,Canada,North America
8,Rahul Kumar,India,Asia


6. Categorize transactions by business quarters using date extraction to identify seasonal shopping patterns.

In [16]:
%%sql
SELECT
    order_id,
    order_date,
    order_amount,
    CASE
        WHEN strftime('%m', order_date) IN ('01', '02', '03') THEN 'Q1 - Winter/Spring'
        WHEN strftime('%m', order_date) IN ('04', '05', '06') THEN 'Q2 - Summer'
        ELSE 'H2 - Rest of Year'
    END AS financial_quarter
FROM online_orders;


 * sqlite://
Done.


order_id,order_date,order_amount,financial_quarter
1,2026-01-15,1200.5,Q1 - Winter/Spring
2,2026-01-18,450.0,Q1 - Winter/Spring
3,2026-02-02,85.0,Q1 - Winter/Spring
4,2026-02-10,45.25,Q1 - Winter/Spring
5,2026-02-14,120.0,Q1 - Winter/Spring
6,2026-03-01,899.0,Q1 - Winter/Spring
7,2026-03-05,60.0,Q1 - Winter/Spring
8,2026-03-12,35.0,Q1 - Winter/Spring


7. Tag products based on their category type to assign appropriate transportation modes and handling priorities.

In [17]:
%%sql
SELECT
    order_id,
    category,
    order_amount,
    CASE
        WHEN category = 'Electronics' THEN 'High Priority Ship'
        WHEN category = 'Furniture' THEN 'Heavy Freight'
        ELSE 'Standard Handling'
    END AS shipping_tier
FROM online_orders;


 * sqlite://
Done.


order_id,category,order_amount,shipping_tier
1,Electronics,1200.5,High Priority Ship
2,Furniture,450.0,Heavy Freight
3,Electronics,85.0,High Priority Ship
4,Apparel,45.25,Standard Handling
5,Apparel,120.0,Standard Handling
6,Furniture,899.0,Heavy Freight
7,Electronics,60.0,High Priority Ship
8,Apparel,35.0,Standard Handling


8. Differentiate between individual consumer retail purchasing and bulk enterprise corporate buying models.

In [18]:
%%sql
SELECT
    order_id,
    customer_name,
    order_amount,
    CASE
        WHEN order_amount > 1000.00 THEN 'Bulk Corporate Order'
        WHEN order_amount BETWEEN 100.00 AND 1000.00 THEN 'Standard Commercial'
        ELSE 'Individual Retail'
    END AS business_model
FROM online_orders;


 * sqlite://
Done.


order_id,customer_name,order_amount,business_model
1,John Doe,1200.5,Bulk Corporate Order
2,Jane Smith,450.0,Standard Commercial
3,Rahul Kumar,85.0,Individual Retail
4,Anna Miller,45.25,Individual Retail
5,John Doe,120.0,Standard Commercial
6,Carlos Ruiz,899.0,Standard Commercial
7,Jane Smith,60.0,Individual Retail
8,Rahul Kumar,35.0,Individual Retail


9. Aggregate individual customer profiles and assign tier status rewards based on their highest single purchase threshold.

In [19]:
%%sql
SELECT
    customer_name,
    CASE
        WHEN MAX(order_amount) >= 1000.00 THEN '🥇 VIP Customer'
        WHEN MAX(order_amount) >= 400.00 THEN '🥈 Preferred Customer'
        ELSE 'Standard Member'
    END AS customer_status
FROM online_orders
GROUP BY customer_name;


 * sqlite://
Done.


customer_name,customer_status
Anna Miller,Standard Member
Carlos Ruiz,🥈 Preferred Customer
Jane Smith,🥈 Preferred Customer
John Doe,🥇 VIP Customer
Rahul Kumar,Standard Member


10. Create a dynamic column to calculate logistics overhead fees inversely matching the overall order basket value.

In [20]:
%%sql
SELECT
    order_id,
    order_amount,
    CASE
        WHEN order_amount >= 500.00 THEN 0.00
        WHEN order_amount >= 100.00 THEN 15.00
        ELSE 25.00
    END AS estimated_shipping_cost
FROM online_orders;


 * sqlite://
Done.


order_id,order_amount,estimated_shipping_cost
1,1200.5,0.0
2,450.0,15.0
3,85.0,25.0
4,45.25,25.0
5,120.0,15.0
6,899.0,0.0
7,60.0,25.0
8,35.0,25.0
